# Final Stable RAG Notebook

## Install Dependencies

In [25]:
!pip install -q transformers sentence-transformers faiss-cpu wikipedia pandas numpy tqdm datasets scikit-learn

## Imports

In [26]:
import json
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
import wikipedia
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


## Configuration

In [27]:
MAX_SAMPLES = 100

CHUNK_SIZE = 100
OVERLAP = 30

TOP_K = 20
TOP_N = 5

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
GENERATION_MODEL = "google/flan-t5-base"


## Load Dataset

In [28]:
DATA_PATH = "NQ-open.dev.jsonl"

def load_jsonl(path, max_samples=None):

    data = []

    with open(path, 'r', encoding='utf-8') as f:

        for idx, line in enumerate(f):

            if max_samples and idx >= max_samples:
                break

            data.append(json.loads(line))

    return pd.DataFrame(data)

df = load_jsonl(DATA_PATH, MAX_SAMPLES)

df.head()


,question,answer
0,when was the last time anyone was on the moon,"[14 December 1972 UTC, December 1972]"
1,who wrote he ain't heavy he's my brother lyrics,"[Bobby Scott, Bob Russell]"
2,how many seasons of the bastard executioner ar...,"[one, one season]"
3,when did the eagles win last super bowl,[2017]
4,who won last year's ncaa women's basketball,[South Carolina]


## Preprocessing

In [29]:
def normalize_answer(ans):

    if isinstance(ans, list):
        return ans[0]

    return ans

df['answer'] = df['answer'].apply(normalize_answer)


## Improved Wikipedia Retrieval

In [30]:
def retrieve_wikipedia_context(query, fallback_results=5):

    documents = []

    try:

        page = wikipedia.page(
            query,
            auto_suggest=True
        )

        documents.append(page.content)

    except Exception:

        try:

            search_results = wikipedia.search(
                query,
                results=fallback_results
            )

            for title in search_results:

                try:

                    page = wikipedia.page(
                        title,
                        auto_suggest=False
                    )

                    documents.append(page.content)

                except Exception:
                    continue

        except Exception:
            pass

    return "\n".join(documents)


In [31]:
contexts = []

for question in tqdm(df['question']):

    context = retrieve_wikipedia_context(question)

    contexts.append(context)

df['context'] = contexts

print("Contexts Retrieved")


100%|██████████| 100/100 [00:59<00:00,  1.68it/s]

Contexts Retrieved


## Chunking

In [32]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = words[start:end]

        chunks.append(" ".join(chunk))

        start += chunk_size - overlap

    return chunks


In [33]:
all_chunks = []

for idx, row in tqdm(df.iterrows(), total=len(df)):

    chunks = chunk_text(row['context'])

    all_chunks.extend(chunks)

print("Total Chunks:", len(all_chunks))


100%|██████████| 100/100 [00:00<00:00, 1997.98it/s]

Total Chunks: 5598


## Embedding Model

In [34]:
embed_model = SentenceTransformer(
    EMBEDDING_MODEL
)

chunk_embeddings = embed_model.encode(
    all_chunks,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True
)

chunk_embeddings = np.array(
    chunk_embeddings
).astype('float32')

print(chunk_embeddings.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/700 [00:00<?, ?it/s]

(5598, 384)


## FAISS Index

In [35]:
embedding_dim = chunk_embeddings.shape[1]

index = faiss.IndexFlatIP(
    embedding_dim
)

index.add(chunk_embeddings)

print("Index Size:", index.ntotal)


Index Size: 5598


## Dense Retrieval

In [36]:
def retrieve_chunks(query, top_k=TOP_K):

    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True
    ).astype('float32')

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved = []

    for idx in indices[0]:

        retrieved.append(all_chunks[idx])

    return retrieved


## CrossEncoder Reranking

In [37]:
reranker = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2'
)

def rerank_chunks(query, chunks, top_n=TOP_N):

    pairs = [[query, chunk] for chunk in chunks]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(chunks, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        chunk
        for chunk, score in ranked[:top_n]
    ]


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Generator Model

In [38]:
tokenizer = AutoTokenizer.from_pretrained(
    GENERATION_MODEL
)

model_gen = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATION_MODEL
)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## Prompt Engineering

In [39]:
def build_prompt(context, question):

    return f'''
Answer ONLY using the provided context.

If the answer is not present,
say:
I don't know.

Context:
{context}

Question:
{question}

Short factual answer:
'''


## Answer Generation

In [40]:
def generate_answer(question):

    retrieved_chunks = retrieve_chunks(
        question,
        top_k=TOP_K
    )

    reranked_chunks = rerank_chunks(
        question,
        retrieved_chunks,
        top_n=TOP_N
    )

    final_context = "\n\n".join(
        reranked_chunks
    )

    prompt = build_prompt(
        final_context,
        question
    )

    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=1024
    )

    outputs = model_gen.generate(
        **inputs,
        max_new_tokens=32,
        temperature=0.0
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, final_context


## Evaluation

In [41]:
def clean_text(text):

    text = text.lower()

    text = re.sub(
        r'[^a-z0-9 ]',
        ' ',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text


In [42]:
def exact_match(prediction, ground_truth):

    pred = clean_text(prediction)

    gt = clean_text(ground_truth)

    return int(pred == gt)


In [43]:
def f1_score(prediction, ground_truth):

    pred_tokens = clean_text(
        prediction
    ).split()

    gt_tokens = clean_text(
        ground_truth
    ).split()

    common = set(pred_tokens) & set(gt_tokens)

    if len(common) == 0:
        return 0

    precision = len(common) / len(pred_tokens)

    recall = len(common) / len(gt_tokens)

    return (
        2 * precision * recall
    ) / (precision + recall)


In [44]:
def retrieval_hit(context, answer):

    context = clean_text(context)

    answer = clean_text(answer)

    return int(answer in context)


## Evaluate Pipeline

In [47]:
results = []

EVAL_SAMPLES = 50

for idx in tqdm(range(EVAL_SAMPLES)):

    question = df.iloc[idx]['question']

    answer = df.iloc[idx]['answer']

    prediction, context = generate_answer(
        question
    )

    em = exact_match(
        prediction,
        answer
    )

    f1 = f1_score(
        prediction,
        answer
    )

    hit = retrieval_hit(
        context,
        answer
    )

    results.append({

        'question': question,
        'ground_truth': answer,
        'prediction': prediction,
        'exact_match': em,
        'f1_score': f1,
        'retrieval_hit': hit
    })

results_df = pd.DataFrame(results)

print("Average Exact Match:",
      results_df['exact_match'].mean())

print("Average F1 Score:",
      results_df['f1_score'].mean())

print("Retrieval Hit Rate:",
      results_df['retrieval_hit'].mean())


100%|██████████| 50/50 [02:47<00:00,  3.35s/it]

Average Exact Match: 0.1
Average F1 Score: 0.16229351155666943
Retrieval Hit Rate: 0.34


## Error Analysis

In [46]:
failed_cases = results_df[
    results_df['exact_match'] == 0
]

failed_cases[
    ['question',
     'ground_truth',
     'prediction']
].head(10)


,question,ground_truth,prediction
0,when was the last time anyone was on the moon,14 December 1972 UTC,1976
1,who wrote he ain't heavy he's my brother lyrics,Bobby Scott,Steve Goodman and John Prine
2,how many seasons of the bastard executioner ar...,one,Six
3,when did the eagles win last super bowl,2017,May 2022
4,who won last year's ncaa women's basketball,South Carolina,I don't know
5,when did the isle of wight become an island,During the last Ice Age,1544
6,love yourself by justin bieber is about who,Rihanna,I don't know
8,what is the hot coffee mod in san andreas,a normally inaccessible mini-game,Hot Coffee is a normally inaccessible mini-gam...
9,what is the maximum data rate for the 802.11a ...,54 Mbit/s,I don't know
10,which state is located in the centre of india,Chhattisgarh,Madhya Pradesh
